# Portfolio Optimization (DBA5106 Group Project 1, Group 38)

This notebook contains the code for the team's report. Running all cells from top to bottom reproduces every table in the report, and the final section confirms that the computed tables match the figures quoted in the report.

| Notebook section | Report section |
|---|---|
| 1. Data preparation | 2. Data and Evaluation Design |
| 2. Shared building blocks | 3.2 and 3.3 (regression set-up) |
| 3. Baseline models | 3.4 Baseline Results, Table 1 |
| 4. Constraints | 4.1, Table 2 |
| 5. Structure and loss screens | 4.2 and 4.4, Tables 3 and 5 |
| 6. Long-only grid | 4.3 and 4.5, Tables 4 and 6 |
| 7. Check against the report | All quoted numbers |

Returns are in percentage points (1.2 means 1.2%). Sharpe ratios are daily and unannualised, with a zero risk-free rate. Transaction costs are not modelled. A full run takes about 25 minutes.

**Data file.** Place the Kenneth French file `100_Portfolios_10x10_Daily.csv` in the same folder as this notebook.

In [1]:
import io
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import brentq
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.model_selection import KFold, TimeSeriesSplit

# LASSO occasionally stops at max_iter for extreme penalties; this does not affect the selected model.
warnings.filterwarnings('ignore', category=ConvergenceWarning)
pd.set_option('display.max_columns', None)

RAW_FILE = Path('100_Portfolios_10x10_Daily.csv')

START_DATE = pd.Timestamp('2026-01-02')   # first portfolio date
JUNE_END = pd.Timestamp('2026-06-30')     # end of the validation period
JULY_END = pd.Timestamp('2026-07-31')     # last available date, used by the long-only grid

WINDOW = 126                              # baseline training window (trading days)
ALPHAS = np.logspace(-8, 8, 101)          # penalty grid for LASSO and Ridge
CV_SPLITS = 5                             # number of cross-validation folds

## 1. Data preparation

The French file holds several daily tables, and only the second one, **Average Equal Weighted Returns -- Daily**, is used. The codes −99.99 and −999 mark missing values. The 90th and 100th portfolios (`ME9 BM10` and `BIG HiBM`) are dropped because of missing data, which leaves 98 assets.

In [2]:
lines = RAW_FILE.read_text().splitlines()

# Locate the equal-weighted table: a title line, a header line, then rows until the next blank line.
start = next(i for i, line in enumerate(lines) if 'Average Equal Weighted Returns -- Daily' in line)
end = next(i for i in range(start + 2, len(lines)) if not lines[i].strip())
table = pd.read_csv(io.StringIO('\n'.join(lines[start + 1:end])), skipinitialspace=True)

table = table.rename(columns={table.columns[0]: 'Date'})
table['Date'] = pd.to_datetime(table['Date'].astype(str), format='%Y%m%d')
table = table.replace([-99.99, -999], np.nan)
table = table.drop(columns=['ME9 BM10', 'BIG HiBM'])   # the 90th and 100th portfolios
data = table.sort_values('Date').reset_index(drop=True)

dates = pd.DatetimeIndex(data['Date'])
asset_names = data.columns[1:].to_numpy(dtype=str)
returns = data.iloc[:, 1:].to_numpy(dtype=float)       # rows = days, columns = 98 assets

print('Days x assets:', returns.shape)
print('Missing values since 2023:', int(np.isnan(returns[dates >= '2023-01-01']).sum()))
data.head()

Days x assets: (26296, 98)
Missing values since 2023: 0


,Date,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,ME1 BM5,ME1 BM6,ME1 BM7,ME1 BM8,ME1 BM9,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,ME2 BM5,ME2 BM6,ME2 BM7,ME2 BM8,ME2 BM9,ME2 BM10,ME3 BM1,ME3 BM2,ME3 BM3,ME3 BM4,ME3 BM5,ME3 BM6,ME3 BM7,ME3 BM8,ME3 BM9,ME3 BM10,ME4 BM1,ME4 BM2,ME4 BM3,ME4 BM4,ME4 BM5,ME4 BM6,ME4 BM7,ME4 BM8,ME4 BM9,ME4 BM10,ME5 BM1,ME5 BM2,ME5 BM3,ME5 BM4,ME5 BM5,ME5 BM6,ME5 BM7,ME5 BM8,ME5 BM9,ME5 BM10,ME6 BM1,ME6 BM2,ME6 BM3,ME6 BM4,ME6 BM5,ME6 BM6,ME6 BM7,ME6 BM8,ME6 BM9,ME6 BM10,ME7 BM1,ME7 BM2,ME7 BM3,ME7 BM4,ME7 BM5,ME7 BM6,ME7 BM7,ME7 BM8,ME7 BM9,ME7 BM10,ME8 BM1,ME8 BM2,ME8 BM3,ME8 BM4,ME8 BM5,ME8 BM6,ME8 BM7,ME8 BM8,ME8 BM9,ME8 BM10,ME9 BM1,ME9 BM2,ME9 BM3,ME9 BM4,ME9 BM5,ME9 BM6,ME9 BM7,ME9 BM8,ME9 BM9,BIG LoBM,ME10 BM2,ME10 BM3,ME10 BM4,ME10 BM5,ME10 BM6,ME10 BM7,ME10 BM8,ME10 BM9
0,1926-07-01,NaN,0.00,NaN,1.59,-3.13,4.32,2.55,8.90,0.00,0.72,-0.65,-1.48,-0.35,1.93,NaN,-1.21,1.16,0.31,0.64,-1.69,0.00,0.05,0.81,1.96,0.02,NaN,0.15,-0.32,0.01,1.05,1.15,0.19,1.72,NaN,-0.16,0.13,-0.71,-0.76,0.08,-0.75,0.76,-1.68,0.18,-0.48,-0.98,-0.81,-0.08,-0.45,-1.56,-1.08,0.03,-0.28,0.82,1.68,0.40,-0.60,0.18,-1.04,-0.62,-0.19,0.01,0.02,-0.26,0.99,-0.15,-0.23,-0.31,-0.23,1.05,0.33,0.67,-0.70,-0.18,0.77,-1.05,0.44,0.55,-0.29,0.69,-1.35,0.14,-0.60,-0.22,0.20,0.03,0.45,-0.59,0.73,0.56,-0.02,0.24,0.11,0.00,0.09,0.51,-0.09,-0.34,0.17
1,1926-07-02,NaN,-0.66,NaN,0.00,-1.42,-4.11,0.30,-0.58,2.82,0.59,0.78,1.22,-1.40,1.30,NaN,-0.97,-1.79,0.30,-1.35,-1.81,9.09,0.14,0.00,0.02,-1.36,NaN,-0.54,0.11,0.48,-0.16,1.05,-1.80,-0.20,NaN,-0.06,-0.46,-0.39,0.25,0.13,0.87,0.86,0.50,-0.10,0.64,-0.13,0.12,-0.29,-0.22,2.68,0.00,0.66,0.85,0.46,-0.37,-0.30,-0.32,0.79,1.70,1.83,-2.31,-0.10,0.23,0.68,-0.12,2.34,-0.03,0.72,0.08,-0.56,0.66,0.33,-0.39,-0.17,0.03,1.34,0.77,0.42,0.59,1.36,2.39,0.78,0.33,0.33,0.03,0.21,0.34,0.88,0.19,0.15,0.43,0.73,1.20,0.81,0.68,0.27,0.24,0.34,0.34
2,1926-07-06,NaN,0.87,NaN,-4.69,2.74,-1.58,0.00,2.36,-5.90,-1.59,0.36,0.00,-1.06,0.01,NaN,0.67,2.22,-0.87,0.57,1.11,-16.67,1.41,0.20,-0.79,0.49,NaN,-0.62,-0.22,0.00,-0.85,-0.97,1.22,1.01,NaN,0.62,-0.19,-0.75,1.21,3.03,0.22,1.87,1.59,0.44,1.58,0.39,1.16,-0.02,0.15,0.24,0.00,0.14,0.07,-0.27,-0.38,0.65,-0.20,0.36,0.06,-0.21,1.44,0.34,-0.03,-0.21,0.63,0.82,0.56,0.04,-0.07,-0.07,-0.33,1.09,-0.31,0.97,-0.14,0.85,0.72,0.38,-0.58,-0.90,-1.33,0.05,0.15,-0.15,0.19,-0.11,0.39,1.52,-1.45,1.19,0.47,0.04,0.67,0.32,-0.19,-0.05,-0.18,-0.51,-0.34
3,1926-07-07,NaN,-1.43,NaN,4.92,0.60,6.07,0.00,-1.86,-0.03,-0.30,-0.45,-3.44,-1.08,-0.87,NaN,1.41,-1.33,1.11,-1.08,-0.40,0.00,0.92,-0.40,-0.75,-1.15,NaN,-0.28,-3.16,-0.19,-0.21,0.13,0.96,-0.74,NaN,-0.89,-0.36,-0.50,-1.89,-3.44,-1.21,0.01,-0.18,0.10,-0.65,-0.13,0.21,0.12,1.00,-0.37,-1.45,-0.04,-0.77,-0.45,-0.59,0.94,0.58,-0.17,2.25,0.27,-0.52,-0.10,0.37,0.01,-0.36,1.09,1.15,0.10,-0.13,-0.17,-1.66,0.22,-0.23,0.35,0.13,-0.21,-0.26,0.08,2.00,-1.13,2.36,0.32,-0.11,-0.18,0.36,-0.09,-0.27,-0.34,-0.02,3.35,-0.18,-0.06,0.89,0.35,0.04,0.16,0.09,0.51,0.17
4,1926-07-08,NaN,0.00,NaN,1.56,-0.59,-0.54,0.00,7.25,0.89,2.52,1.12,-1.48,-1.45,0.51,NaN,-3.73,-0.60,-0.37,-0.63,1.75,0.00,0.56,0.00,0.35,-0.04,NaN,0.00,1.66,-0.94,-0.52,0.73,0.20,1.68,NaN,0.67,-0.07,-0.37,0.36,-0.91,-0.10,1.39,-0.21,-1.38,0.33,-0.71,0.94,0.50,-0.57,1.69,0.00,-0.44,0.93,1.29,-0.05,-0.78,0.41,-0.73,-0.01,-0.46,0.39,-0.10,0.91,1.27,0.81,0.71,0.01,0.19,0.25,1.20,1.68,-0.04,0.16,-0.53,0.14,-0.12,0.93,-0.23,2.14,0.92,2.31,1.51,-0.11,0.01,-0.01,-0.22,0.40,0.90,0.34,0.00,0.64,-0.05,1.14,0.71,0.01,0.20,0.30,0.34,0.51


## 2. Shared building blocks

**Regression set-up (report Section 3.2).** For a window of returns $R$, the target is the Equal Weight return $y = Rw_{EW}$ and the features are $X = RN$ with $N = [I_{p-1}; -\mathbf{1}^T]$. A fitted $\beta$ becomes portfolio weights through $w = w_{EW} - N\beta$. The columns of $N$ sum to zero, so the weights always sum to one.

**Long-only adjustment (Sections 4.3 and 4.5).** If a portfolio has negative weights, it is moved towards Equal Weight just far enough that the most negative weight reaches zero.

**Sharpe Ridge (Sections 4.3 to 4.5).** Same Ridge regression, but the penalty is chosen by the Sharpe ratio of the portfolio returns on the five chronological validation folds instead of by mean squared error.

**Summary statistics.** Sharpe ratio, cumulative return, daily SD, and averages of gross leverage, turnover and HHI (concentration).

In [3]:
def regression_setup(R):
    '''Return y = R w_EW, X = R N, the Equal Weight vector and N for a window of returns R.'''
    p = R.shape[1]
    w_ew = np.full(p, 1 / p)
    N = np.vstack([np.eye(p - 1), -np.ones((1, p - 1))])
    return R @ w_ew, R @ N, w_ew, N


def make_long_only(w):
    '''Move w towards Equal Weight only as far as needed to remove negative weights.'''
    if w.min() >= 0:
        return w.copy()
    w_ew = np.full(len(w), 1 / len(w))
    negative = w < 0
    step = np.min(w_ew[negative] / (w_ew[negative] - w[negative]))   # largest step that keeps all weights >= 0
    w_new = w_ew + step * (w - w_ew)
    w_new[np.abs(w_new) < 1e-14] = 0                                # remove rounding noise
    w_new = np.maximum(w_new, 0)
    return w_new / w_new.sum()


def fit_lasso(R, long_only=False):
    '''LASSO with the penalty chosen by chronological cross-validation (MSE).'''
    y, X, w_ew, N = regression_setup(R)
    model = LassoCV(alphas=ALPHAS, cv=TimeSeriesSplit(CV_SPLITS), fit_intercept=True,
                    max_iter=100_000, tol=1e-4, n_jobs=-1, selection='cyclic').fit(X, y)
    w = w_ew - N @ model.coef_
    return make_long_only(w) if long_only else w


def fit_sharpe_ridge(R, long_only=True):
    '''Ridge whose penalty maximises the Sharpe ratio of the pooled validation-fold returns.'''
    fold_returns = []
    for train, validation in TimeSeriesSplit(CV_SPLITS).split(R):
        y, X, w_ew, N = regression_setup(R[train])
        X, y = X - X.mean(axis=0), y - y.mean()   # centring = fitting an intercept
        # One eigen-decomposition gives the Ridge solution for all 101 penalties at once.
        eigenvalues, eigenvectors = np.linalg.eigh(X.T @ X)
        rhs = eigenvectors.T @ (X.T @ y)
        returns_per_alpha = []
        for alpha in ALPHAS:
            beta = eigenvectors @ (rhs / (eigenvalues + alpha))
            w = w_ew - N @ beta
            if long_only:
                w = make_long_only(w)
            returns_per_alpha.append(R[validation] @ w)
        fold_returns.append(np.array(returns_per_alpha))

    validation_returns = np.concatenate(fold_returns, axis=1)      # rows = penalties, columns = validation days
    sd = validation_returns.std(axis=1, ddof=1)
    negative_sharpe = np.divide(-validation_returns.mean(axis=1), sd,
                                out=np.full(len(ALPHAS), np.inf), where=sd > np.finfo(float).eps)
    # If several penalties tie, prefer the strongest regularisation.
    best = np.flatnonzero(np.isclose(negative_sharpe, negative_sharpe.min(), rtol=1e-12, atol=1e-12))[-1]
    alpha = ALPHAS[best]

    y, X, w_ew, N = regression_setup(R)
    X, y = X - X.mean(axis=0), y - y.mean()
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    w = w_ew - N @ beta
    return make_long_only(w) if long_only else w


def portfolio_record(date, model, w, next_day_returns, previous_w, **labels):
    '''One row of daily results: realised return and weight diagnostics.'''
    turnover = np.nan if previous_w is None else 0.5 * np.abs(w - previous_w).sum()
    return dict(Date=date, Model=model, ReturnPercent=float(w @ next_day_returns),
                GrossLeverage=float(np.abs(w).sum()), HHI=float(w @ w),
                MaxAbsWeight=float(np.abs(w).max()), Turnover=turnover, **labels)


def summarize(daily, by):
    '''Sharpe, cumulative return, daily SD and average diagnostics for each group.'''
    rows = []
    for key, group in daily.groupby(by, sort=False):
        r = group['ReturnPercent'].to_numpy()
        sd = r.std(ddof=1)
        rows.append(dict(zip(by, key if isinstance(key, tuple) else (key,)),
                         Sharpe=r.mean() / sd, CumulativeReturnPct=100 * (np.prod(1 + r / 100) - 1),
                         MeanDailyReturnPct=r.mean(), DailySDPct=sd,
                         GrossLeverage=group['GrossLeverage'].mean(), Turnover=group['Turnover'].mean(),
                         HHI=group['HHI'].mean(), MaxAbsWeight=group['MaxAbsWeight'].mean()))
    return pd.DataFrame(rows)

## 3. Baseline models (report Section 3, Table 1)

For each trading day from 2 January to 31 July 2026, this loop

1. takes the previous 126 days of returns $R$,
2. builds $y$ and $X$,
3. estimates $\beta$ with OLS (MinVar), cross-validated LASSO and cross-validated Ridge,
4. converts each $\beta$ to weights with $w = w_{EW} - N\beta$,
5. applies the weights to the next day and saves the return.

LASSO and Ridge pick their penalty from 101 values using five chronological folds and mean squared error. Short positions and leverage are allowed. Table 1 uses January to June; the July figures are only used for the debugging check.

In [4]:
evaluation_days = np.flatnonzero((dates >= START_DATE) & (dates <= JULY_END))

records, previous = [], {}
for t in evaluation_days:
    R = returns[t - WINDOW:t]                       # previous 126 trading days
    y, X, w_ew, N = regression_setup(R)

    betas = {}
    ols = np.linalg.lstsq(np.column_stack([np.ones(len(y)), X]), y, rcond=None)[0]
    betas['MinVar'] = ols[1:]                       # the intercept does not enter the weights
    betas['LASSO'] = LassoCV(alphas=ALPHAS, cv=TimeSeriesSplit(CV_SPLITS), fit_intercept=True,
                             max_iter=100_000, tol=1e-4, n_jobs=-1, selection='cyclic').fit(X, y).coef_
    betas['Ridge'] = RidgeCV(alphas=ALPHAS, cv=TimeSeriesSplit(CV_SPLITS), fit_intercept=True,
                             scoring='neg_mean_squared_error').fit(X, y).coef_

    weights = {'EqualWeight': w_ew}
    for name, beta in betas.items():
        weights[name] = w_ew - N @ beta             # beta -> portfolio weights

    for name, w in weights.items():
        records.append(portfolio_record(dates[t], name, w, returns[t], previous.get(name)))
        previous[name] = w

baseline_daily = pd.DataFrame(records)
table1 = summarize(baseline_daily[baseline_daily['Date'] <= JUNE_END], ['Model'])
print('Table 1: baseline validation results, 2 January to 30 June 2026')
display(table1[['Model', 'MeanDailyReturnPct', 'DailySDPct', 'Sharpe', 'CumulativeReturnPct']].round(4))

Table 1: baseline validation results, 2 January to 30 June 2026


,Model,MeanDailyReturnPct,DailySDPct,Sharpe,CumulativeReturnPct
0,EqualWeight,0.1322,0.9974,0.1325,16.9328
1,MinVar,-0.0151,0.8770,-0.0172,-2.2966
2,LASSO,0.0424,0.5471,0.0775,5.1567
3,Ridge,-0.0125,0.5841,-0.0215,-1.7350


### Debugging checks

The project instructions give reference numbers to confirm the set-up. All of them are reproduced here.

In [5]:
first = evaluation_days[0]
R6 = returns[first - WINDOW:first, :6]                       # first six assets only
y6, X6, w_ew6, N6 = regression_setup(R6)
beta6 = np.linalg.lstsq(np.column_stack([np.ones(len(y6)), X6]), y6, rcond=None)[0][1:]
w6 = w_ew6 - N6 @ beta6
july_sharpe = summarize(baseline_daily, ['Model']).set_index('Model')['Sharpe']

checks = pd.DataFrame({
    'Check': ['Row of 2026-01-02 in the full table', 'Six-asset MinVar weights',
              'Six-asset return on 2026-01-02 (%)', 'Equal Weight Sharpe, Jan-Jul', 'MinVar Sharpe, Jan-Jul'],
    'Computed': [first + 1, np.round(w6, 3).tolist(), round(w6 @ returns[first, :6], 3),
                 round(july_sharpe['EqualWeight'], 3), round(july_sharpe['MinVar'], 3)],
    'Reference': [26152, [-0.111, -0.199, 0.348, 0.46, 0.006, 0.496], 1.393, 0.111, 0.021]})
display(checks)

assert first + 1 == 26152
np.testing.assert_allclose(w6, [-0.111, -0.199, 0.348, 0.46, 0.006, 0.496], atol=0.0005)
np.testing.assert_allclose(w6 @ returns[first, :6], 1.393, atol=0.0005)
np.testing.assert_allclose([july_sharpe['EqualWeight'], july_sharpe['MinVar']], [0.111, 0.021], atol=0.0005)

,Check,Computed,Reference
0,Row of 2026-01-02 in the full table,26152,26152
1,Six-asset MinVar weights,"[-0.111, -0.199, 0.348, 0.46, 0.006, 0.496]","[-0.111, -0.199, 0.348, 0.46, 0.006, 0.496]"
2,Six-asset return on 2026-01-02 (%),1.393,1.393
3,"Equal Weight Sharpe, Jan-Jul",0.111,0.111
4,"MinVar Sharpe, Jan-Jul",0.021,0.021


## 4. Constraints (report Section 4.1, Table 2)

Each day, LASSO and Ridge are fitted and each portfolio $w_0$ is moved to the **closest** portfolio that satisfies a constraint while still summing to one, $w^* = \arg\min_w \|w - w_0\|^2$.

- **No short selling** ($w_i \ge 0$) uses a projection onto the simplex.
- **Maximum weight** ($|w_i| \le c$) shifts all weights by the same amount and clips them to $[-c, c]$.
- **Gross leverage** ($\sum|w_i| \le L$) shrinks all weights towards zero by the same amount (soft thresholding).

This experiment uses 21 penalties and standard (unshuffled) five-fold cross-validation, with scikit-learn's default scoring. Because these settings differ from Section 3, the unconstrained values are only internal references.

In [6]:
CONSTRAINT_ALPHAS = np.logspace(-8, 8, 21)


def project_no_short(w0):
    '''Closest weights with w >= 0 and sum 1 (projection onto the simplex).'''
    sorted_w = np.sort(w0)[::-1]
    shifts = (np.cumsum(sorted_w) - 1) / np.arange(1, len(w0) + 1)
    shift = shifts[np.flatnonzero(sorted_w > shifts)[-1]]
    return np.maximum(w0 - shift, 0)


def project_max_weight(w0, cap):
    '''Closest weights with |w_i| <= cap and sum 1.'''
    shift = brentq(lambda a: np.clip(w0 - a, -cap, cap).sum() - 1,
                   w0.min() - cap - 1, w0.max() + cap + 1, xtol=1e-13)
    return np.clip(w0 - shift, -cap, cap)


def project_leverage(w0, limit):
    '''Closest weights with sum |w_i| <= limit and sum 1.'''
    if np.abs(w0).sum() <= limit:
        return w0.copy()

    def shrink(threshold):
        # For a given amount of shrinkage, choose the shift that keeps the weights summing to one.
        def soft(a):
            x = w0 - a
            return np.sign(x) * np.maximum(np.abs(x) - threshold, 0)
        a = brentq(lambda a: soft(a).sum() - 1, w0.min() - threshold - 2, w0.max() + threshold + 2, xtol=1e-13)
        return soft(a)

    upper = max(1, np.abs(w0).max())
    while np.abs(shrink(upper)).sum() > limit:
        upper *= 2
    threshold = brentq(lambda s: np.abs(shrink(s)).sum() - limit, 0, upper, xtol=1e-12)
    return shrink(threshold)


CONSTRAINTS = {
    'Unconstrained': lambda w: w.copy(),
    'No short selling': project_no_short,
    'Max weight 5%': lambda w: project_max_weight(w, 0.05),
    'Max weight 10%': lambda w: project_max_weight(w, 0.10),
    'Max weight 20%': lambda w: project_max_weight(w, 0.20),
    'Leverage <= 1.5': lambda w: project_leverage(w, 1.5),
    'Leverage <= 2': lambda w: project_leverage(w, 2.0),
    'Leverage <= 3': lambda w: project_leverage(w, 3.0),
}

validation_days = np.flatnonzero((dates >= START_DATE) & (dates <= JUNE_END))
records, previous = [], {}
for t in validation_days:
    y, X, w_ew, N = regression_setup(returns[t - WINDOW:t])
    fitted = {
        'LASSO': LassoCV(alphas=CONSTRAINT_ALPHAS, cv=KFold(5, shuffle=False), fit_intercept=True,
                         max_iter=100_000, tol=1e-6, n_jobs=-1, selection='cyclic').fit(X, y).coef_,
        'Ridge': RidgeCV(alphas=CONSTRAINT_ALPHAS, cv=KFold(5, shuffle=False), fit_intercept=True).fit(X, y).coef_,
    }
    for model, beta in fitted.items():
        w0 = w_ew - N @ beta
        for constraint, project in CONSTRAINTS.items():
            w = project(w0)
            key = (model, constraint)
            records.append(portfolio_record(dates[t], model, w, returns[t], previous.get(key), Constraint=constraint))
            previous[key] = w

constraint_daily = pd.DataFrame(records)
table2 = summarize(constraint_daily, ['Model', 'Constraint'])
print('Table 2 (all constraints): 2 January to 30 June 2026')
display(table2[['Model', 'Constraint', 'Sharpe', 'CumulativeReturnPct', 'GrossLeverage', 'MaxAbsWeight']].round(4))

Table 2 (all constraints): 2 January to 30 June 2026


,Model,Constraint,Sharpe,CumulativeReturnPct,GrossLeverage,MaxAbsWeight
0,LASSO,Unconstrained,0.0273,1.6357,3.1669,0.4372
1,LASSO,No short selling,0.0996,7.3173,1.0000,0.4000
2,LASSO,Max weight 5%,0.1164,11.0662,2.2727,0.0500
3,LASSO,Max weight 10%,0.1051,8.6843,2.7835,0.1000
4,LASSO,Max weight 20%,0.0891,6.3829,3.0250,0.2000
5,LASSO,Leverage <= 1.5,0.1017,7.0553,1.5000,0.4224
6,LASSO,Leverage <= 2,0.0901,5.9492,2.0000,0.4296
7,LASSO,Leverage <= 3,0.0379,2.3268,2.9937,0.4361
8,Ridge,Unconstrained,-0.0159,-1.3246,4.2013,0.2094
9,Ridge,No short selling,0.1369,11.8283,1.0000,0.1679


## 5. Structure and loss screens (report Sections 4.2 and 4.4, Tables 3 and 5)

These screens use four half-year **discovery blocks** before 2026, so no 2026 data is involved. Each day the models are refitted on the previous 126 days. The Sharpe ratio is computed for each block, and the median and the worst of the four blocks are reported. Leverage, turnover and HHI are averaged within each block, and their medians are compared with Ridge.

- **Ridge (reference)** chooses its penalty by mean squared error.
- **Additive Structured Ridge (4.2)** models each asset's adjustment as a size effect plus a book-to-market effect, which leaves 18 free directions instead of 97.
- **Ridge / EW 50/50 (4.2)** sits halfway between Ridge and Equal Weight.
- **Sharpe-selected Ridge (4.4)** chooses its penalty by validation Sharpe instead of MSE.

Short positions are allowed throughout this section.

In [7]:
def additive_basis(names):
    '''18 orthonormal, zero-sum directions for additive size and book-to-market effects.'''
    size = [1 if n.split()[0] == 'SMALL' else 10 if n.split()[0] == 'BIG' else int(n.split()[0][2:]) for n in names]
    bm = [1 if n.split()[1] == 'LoBM' else 10 if n.split()[1] == 'HiBM' else int(n.split()[1][2:]) for n in names]
    indicators = np.zeros((len(names), 20))                   # 10 size dummies + 10 book-to-market dummies
    indicators[np.arange(len(names)), np.array(size) - 1] = 1
    indicators[np.arange(len(names)), 10 + np.array(bm) - 1] = 1
    centred = indicators - indicators.mean(axis=0)            # centred columns keep the weights summing to one
    u, s, _ = np.linalg.svd(centred, full_matrices=False)
    rank = int(np.sum(s > max(centred.shape) * s.max() * np.finfo(float).eps))
    return u[:, :rank]


B = additive_basis(asset_names)
assert B.shape == (98, 18)

DISCOVERY_BLOCKS = {'2023-H2': ('2023-07-01', '2023-12-31'), '2024-H1': ('2024-01-01', '2024-06-30'),
                    '2024-H2': ('2024-07-01', '2024-12-31'), '2025-H1': ('2025-01-01', '2025-06-30')}


def ridge_mse_cv(X, y):
    return RidgeCV(alphas=ALPHAS, cv=TimeSeriesSplit(CV_SPLITS), fit_intercept=True,
                   scoring='neg_mean_squared_error').fit(X, y).coef_


records = []
for block, (block_start, block_end) in DISCOVERY_BLOCKS.items():
    previous = {}                                             # turnover restarts at each block
    for t in np.flatnonzero((dates >= block_start) & (dates <= block_end)):
        R = returns[t - WINDOW:t]
        y, X, w_ew, N = regression_setup(R)
        ridge = w_ew - N @ ridge_mse_cv(X, y)
        weights = {
            'Ridge (reference)': ridge,
            'Additive Structured Ridge': w_ew - B @ ridge_mse_cv(R @ B, y),
            'Ridge / EW 50/50': 0.5 * ridge + 0.5 * w_ew,
            'Sharpe-selected Ridge': fit_sharpe_ridge(R, long_only=False),
        }
        for name, w in weights.items():
            records.append(portfolio_record(dates[t], name, w, returns[t], previous.get(name), Block=block))
            previous[name] = w
    print('Finished block', block)

historical_daily = pd.DataFrame(records)
blocks = summarize(historical_daily, ['Block', 'Model'])
screen = blocks.groupby('Model', sort=False).agg(
    MedianSharpe=('Sharpe', 'median'), WorstBlockSharpe=('Sharpe', 'min'),
    GrossLeverage=('GrossLeverage', 'median'), Turnover=('Turnover', 'median'), HHI=('HHI', 'median'))
for metric in ['GrossLeverage', 'Turnover', 'HHI']:
    screen[metric + 'ChangePct'] = 100 * (screen[metric] / screen.loc['Ridge (reference)', metric] - 1)

print('Table 3: structure screen across the four discovery blocks')
display(screen.loc[['Ridge (reference)', 'Additive Structured Ridge', 'Ridge / EW 50/50']].round(4))
print('Table 5: loss screen across the four discovery blocks')
display(screen.loc[['Ridge (reference)', 'Sharpe-selected Ridge'], ['MedianSharpe', 'GrossLeverageChangePct']].round(4))

Finished block 2023-H2


Finished block 2024-H1


Finished block 2024-H2


Finished block 2025-H1
Table 3: structure screen across the four discovery blocks


,MedianSharpe,WorstBlockSharpe,GrossLeverage,Turnover,HHI,GrossLeverageChangePct,TurnoverChangePct,HHIChangePct
Model,,,,,,,,
Ridge (reference),0.0666,-0.0209,4.4462,0.2632,0.3345,0.0000,0.0000,0.0000
Additive Structured Ridge,0.0824,0.0122,3.4229,0.1421,0.2068,-23.0155,-46.0246,-38.1831
Ridge / EW 50/50,0.0575,0.0002,2.3173,0.1316,0.0913,-47.8808,-50.0000,-72.7124


Table 5: loss screen across the four discovery blocks


,MedianSharpe,GrossLeverageChangePct
Model,,
Ridge (reference),0.0666,0.0000
Sharpe-selected Ridge,-0.0223,102.3871


## 6. Long-only grid (report Sections 4.3 and 4.5, Tables 4 and 6)

Seed 100 fixes one random ordering of the 98 assets. The universes of 20, 40, 60 and 80 assets are the first 20, 40, 60 and 80 assets in that ordering, so each is contained in the next. For every training window (126, 189 and 252 days) and universe, the notebook fits the following each day.

- Equal Weight, long-only LASSO and long-only Sharpe Ridge, and
- blends with Equal Weight at 50/50 and 75/25 (75% model, 25% Equal Weight).

That is 3 windows x 5 universes x 7 strategies = 105 configurations, evaluated from 2 January to 31 July 2026. Equal Weight does not use the training window, so it repeats across windows. This cell takes most of the run time.

In [8]:
WINDOWS = (126, 189, 252)
ASSET_COUNTS = (20, 40, 60, 80, 98)
order = np.random.default_rng(100).permutation(len(asset_names))
universes = {k: np.sort(order[:k]) for k in ASSET_COUNTS}   # keep the original column order within each universe

records = []
for window in WINDOWS:
    for k, assets in universes.items():
        previous = {}
        for t in evaluation_days:
            R = returns[t - window:t, assets]
            w_ew = np.full(k, 1 / k)
            weights = {'EqualWeight': w_ew,
                       'LASSO': fit_lasso(R, long_only=True),
                       'SharpeRidge': fit_sharpe_ridge(R, long_only=True)}
            for model in ['LASSO', 'SharpeRidge']:
                weights[f'{model} / EW 50/50'] = 0.50 * weights[model] + 0.50 * w_ew
                weights[f'{model} / EW 75/25'] = 0.75 * weights[model] + 0.25 * w_ew
            for name, w in weights.items():
                assert w.min() >= -1e-12 and abs(w.sum() - 1) < 1e-10      # long-only and fully invested
                records.append(portfolio_record(dates[t], name, w, returns[t, assets], previous.get(name),
                                                WindowDays=window, AssetCount=k))
                previous[name] = w
        print(f'Finished {window}-day window, {k} assets')

grid_daily = pd.DataFrame(records)
grid = summarize(grid_daily, ['WindowDays', 'AssetCount', 'Model'])
columns = ['Model', 'Sharpe', 'CumulativeReturnPct', 'HHI', 'DailySDPct', 'Turnover']

print('Table 4: 126-day window, 20 and 98 assets')
display(grid[(grid.WindowDays == 126) & grid.AssetCount.isin([20, 98])][['AssetCount'] + columns].round(4))
print('Section 4.3.2: 98 assets, all windows')
display(grid[(grid.AssetCount == 98) & grid.Model.isin(['LASSO', 'SharpeRidge'])][['WindowDays'] + columns].round(4))
print('Table 6: 60 assets, 189-day window')
display(grid[(grid.WindowDays == 189) & (grid.AssetCount == 60)][columns].round(4))

Finished 126-day window, 20 assets


Finished 126-day window, 40 assets


Finished 126-day window, 60 assets


Finished 126-day window, 80 assets


Finished 126-day window, 98 assets


Finished 189-day window, 20 assets


... (10 outputs left out)

Section 4.3.2: 98 assets, all windows


,WindowDays,Model,Sharpe,CumulativeReturnPct,HHI,DailySDPct,Turnover
29,126,LASSO,0.1116,15.2776,0.0118,0.9159,0.0160
30,126,SharpeRidge,0.1090,14.7105,0.0114,0.9064,0.0350
64,189,LASSO,0.1141,15.4671,0.0125,0.9053,0.0135
65,189,SharpeRidge,0.1127,15.0400,0.0117,0.8928,0.0171
99,252,LASSO,0.1106,15.0634,0.0124,0.9126,0.0140
100,252,SharpeRidge,0.1108,14.7002,0.0132,0.8895,0.0247


Table 6: 60 assets, 189-day window


,Model,Sharpe,CumulativeReturnPct,HHI,DailySDPct,Turnover
49,EqualWeight,0.1205,17.5127,0.0167,0.9623,0.0000
50,LASSO,0.1202,16.1790,0.0225,0.8936,0.0167
51,SharpeRidge,0.1394,14.6128,0.1960,0.6923,0.0184
52,LASSO / EW 50/50,0.1204,16.8457,0.0181,0.9275,0.0083
53,LASSO / EW 75/25,0.1203,16.5123,0.0199,0.9104,0.0125
54,SharpeRidge / EW 50/50,0.1315,16.0954,0.0615,0.8078,0.0092
55,SharpeRidge / EW 75/25,0.1362,15.3621,0.1175,0.7439,0.0138


## 7. Check against the report

This section confirms that the tables above match the figures quoted in the report. A figure matches when the computed value rounds to the reported value.

In [9]:
results = []

def check(section, item, reported, computed, decimals=4):
    matches = abs(computed - reported) <= 0.5 * 10 ** -decimals + 1e-9
    results.append(dict(Section=section, Item=item, Reported=reported, Computed=round(float(computed), 6), Matches=matches))


# Table 1
t1 = table1.set_index('Model')
for model, sharpe, cum, mean, sd in [('EqualWeight', .1325, 16.93, .1322, .9974), ('MinVar', -.0172, -2.30, -.0151, .8770),
                                     ('LASSO', .0775, 5.16, .0424, .5471), ('Ridge', -.0215, -1.74, -.0125, .5841)]:
    check('3.4', f'{model} Sharpe', sharpe, t1.loc[model, 'Sharpe'])
    check('3.4', f'{model} cumulative %', cum, t1.loc[model, 'CumulativeReturnPct'], 2)
    check('3.4', f'{model} mean daily %', mean, t1.loc[model, 'MeanDailyReturnPct'])
    check('3.4', f'{model} daily SD %', sd, t1.loc[model, 'DailySDPct'])

# Table 2 and Section 4.1 text
t2 = table2.set_index(['Model', 'Constraint'])
for model, constraint, sharpe, cum, lev in [('Ridge', 'Unconstrained', -.0159, -1.32, 4.20), ('Ridge', 'No short selling', .1369, 11.83, 1.00),
                                            ('Ridge', 'Leverage <= 1.5', .1255, 10.11, 1.50), ('LASSO', 'Unconstrained', .0273, 1.64, 3.17),
                                            ('LASSO', 'Max weight 5%', .1164, 11.07, 2.27), ('LASSO', 'Leverage <= 1.5', .1017, 7.06, 1.50)]:
    check('4.1', f'{model}, {constraint}, Sharpe', sharpe, t2.loc[(model, constraint), 'Sharpe'])
    check('4.1', f'{model}, {constraint}, cumulative %', cum, t2.loc[(model, constraint), 'CumulativeReturnPct'], 2)
    check('4.1', f'{model}, {constraint}, gross leverage', lev, t2.loc[(model, constraint), 'GrossLeverage'], 2)
for model, constraint, sharpe in [('Ridge', 'Leverage <= 3', .0487), ('Ridge', 'Leverage <= 2', .1038), ('LASSO', 'No short selling', .0996)]:
    check('4.1', f'{model}, {constraint}, Sharpe', sharpe, t2.loc[(model, constraint), 'Sharpe'])
check('4.1', 'LASSO largest position %', 43.7, 100 * t2.loc[('LASSO', 'Unconstrained'), 'MaxAbsWeight'], 1)

# Tables 3 and 5
for model, median, worst in [('Ridge (reference)', .0666, -.0209), ('Additive Structured Ridge', .0824, .0122), ('Ridge / EW 50/50', .0575, .0002)]:
    check('4.2', f'{model} median Sharpe', median, screen.loc[model, 'MedianSharpe'])
    check('4.2', f'{model} worst-block Sharpe', worst, screen.loc[model, 'WorstBlockSharpe'])
for metric, value in [('GrossLeverage', -23.0), ('Turnover', -46.0), ('HHI', -38.2)]:
    check('4.2', f'Structured Ridge {metric} change %', value, screen.loc['Additive Structured Ridge', metric + 'ChangePct'], 1)
check('4.4', 'Sharpe-selected Ridge median Sharpe', -.0223, screen.loc['Sharpe-selected Ridge', 'MedianSharpe'])
check('4.4', 'Sharpe-selected Ridge leverage change %', 102.4, screen.loc['Sharpe-selected Ridge', 'GrossLeverageChangePct'], 1)

# Table 4 and Section 4.3.2
g = grid.set_index(['WindowDays', 'AssetCount', 'Model'])
table4 = {20: [('EqualWeight', .1217, 18.81, 1.0194), ('LASSO', .1129, 15.84, .9367), ('SharpeRidge', .1220, 18.61, 1.0059),
               ('LASSO / EW 50/50', .1178, 17.32, .9756), ('LASSO / EW 75/25', .1155, 16.58, .9555),
               ('SharpeRidge / EW 50/50', .1219, 18.71, 1.0122), ('SharpeRidge / EW 75/25', .1220, 18.66, 1.0090)],
          98: [('EqualWeight', .1114, 15.95, .9576), ('LASSO', .1116, 15.28, .9159), ('SharpeRidge', .1090, 14.71, .9064),
               ('LASSO / EW 50/50', .1115, 15.61, .9366), ('LASSO / EW 75/25', .1116, 15.45, .9262),
               ('SharpeRidge / EW 50/50', .1103, 15.33, .9314), ('SharpeRidge / EW 75/25', .1097, 15.02, .9187)]}
for k, rows in table4.items():
    for model, sharpe, cum, sd in rows:
        check('4.3', f'{k} assets, {model}, Sharpe', sharpe, g.loc[(126, k, model), 'Sharpe'])
        check('4.3', f'{k} assets, {model}, cumulative %', cum, g.loc[(126, k, model), 'CumulativeReturnPct'], 2)
        check('4.3', f'{k} assets, {model}, daily SD %', sd, g.loc[(126, k, model), 'DailySDPct'])
for window, model, sharpe in [(189, 'LASSO', .1141), (189, 'SharpeRidge', .1127), (252, 'LASSO', .1106), (252, 'SharpeRidge', .1108)]:
    check('4.3', f'98 assets, {window} days, {model}, Sharpe', sharpe, g.loc[(window, 98, model), 'Sharpe'])
for window, turnover in [(126, .0350), (189, .0171)]:
    check('4.3', f'98 assets, {window} days, SharpeRidge turnover', turnover, g.loc[(window, 98, 'SharpeRidge'), 'Turnover'])

# Table 6 and Section 4.5 text
for model, sharpe, cum, hhi, sd in [('EqualWeight', .1205, 17.51, .0167, .9623), ('LASSO', .1202, 16.18, .0225, .8936),
                                    ('LASSO / EW 50/50', .1204, 16.85, .0181, .9275), ('LASSO / EW 75/25', .1203, 16.51, .0199, .9104),
                                    ('SharpeRidge', .1394, 14.61, .1960, .6923), ('SharpeRidge / EW 50/50', .1315, 16.10, .0615, .8078),
                                    ('SharpeRidge / EW 75/25', .1362, 15.36, .1175, .7439)]:
    check('4.5', f'{model} Sharpe', sharpe, g.loc[(189, 60, model), 'Sharpe'])
    check('4.5', f'{model} cumulative %', cum, g.loc[(189, 60, model), 'CumulativeReturnPct'], 2)
    check('4.5', f'{model} HHI', hhi, g.loc[(189, 60, model), 'HHI'])
    check('4.5', f'{model} daily SD %', sd, g.loc[(189, 60, model), 'DailySDPct'])
hhi_cut = 100 * (1 - g.loc[(189, 60, 'SharpeRidge / EW 50/50'), 'HHI'] / g.loc[(189, 60, 'SharpeRidge'), 'HHI'])
check('4.5', '50/50 blend HHI reduction %', 68.6, hhi_cut, 1)

report_check = pd.DataFrame(results)
display(report_check.groupby('Section')['Matches'].agg(['sum', 'count']).rename(columns={'sum': 'matched', 'count': 'checked'}))
display(report_check[~report_check['Matches']])
assert report_check['Matches'].all(), 'Some report values were not reproduced'
print(f'All {len(report_check)} numbers quoted in the report are reproduced.')

,matched,checked
Section,,
3.4,16,16
4.1,22,22
4.2,9,9
4.3,48,48
4.4,2,2
4.5,29,29


,Section,Item,Reported,Computed,Matches


All 126 numbers quoted in the report are reproduced.
